# Axisymmetric Representations of Structure Family Geometries

Right-half cross-section table for the structure family geometries
treated as solids of revolution about the vertical (z) axis. Each row
shows the merged outline of structure + pillar in 2D, suitable as direct
input to an FEA preprocessor that revolves the closed half-profile about
x = 0 to obtain the 3D solid.

**Shape set in this notebook**

7 shapes: ellipsoids (3 variants), cones, caps, cap_cones, doublecones.
Excludes caps_div and caps_shell from the previous (full-profile) table.

**Pillar geometry**

Every structure carries a vertical pillar attached at (0, 0):

- Pillar lateral semi-radius: $R_{xy,p} = 25\,\mu$m
- Pillar height: $H_p = 1.25\,R_z$ (where $R_z$ is the structure's
  vertical semi-axis), or $H_p = 1.25\,(H/2)$ for shapes parameterized
  by $H$ instead of $R_z$ (cones, doublecones)
- Pillar occupies $x \in [0, R_{xy,p}]$, $z \in [-H_p, 0]$ (right half)

For ellipsoids and doublecones (centered at origin so structure occupies
$z \in [-R_z, +R_z]$ or $z \in [-H/2, +H/2]$), the upper portion of
the pillar is volumetrically inside the structure body and is hidden
from the merged exterior outline. For caps, cones, and cap_cones
(structure base at $z=0$, body above), the pillar meets the flat base
in a corner transition with no overlap.

**Cell layout**

1. Imports
2. **Shape generators** (FEA-portable; isolated)
3. Image-loader helpers (crop + black-background masking)
4. Geometry metadata (structure dims and pillar dims per row)
5. Figure builder
6. Save full version (7 rows)
7. Save two-page version

**Conventions used by the generators**

- Each generator returns `(x, z)` describing the right-half merged
  outline as a polyline starting at `(0, z_bottom)` and ending at
  `(0, z_top)`. The polygon is implicitly closed by the axis edge at
  $x = 0$ (suitable for FEA solid-of-revolution input).
- Ellipsoids: centered at origin, $z \in [-R_z, +R_z]$.
- Doublecones: centered at origin, $z \in [-H/2, +H/2]$.
- Cones, caps, cap_cones: base at $z = 0$, apex above.
- Pillars: $x \in [0, R_{xy,p}]$, $z \in [-H_p, 0]$.
- The axis-of-revolution dashed annotation is added in the plotting
  cell only, not in the generator output.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


In [ ]:
# ────────────────────────────────────────────────────────────────────
# FEA-portable axisymmetric shape generators (right-half + pillar).
#
# Output convention
# -----------------
# Each function returns (x, z) for the right-half merged outline of
# (structure ∪ pillar) as a polyline:
#   - Starts at (0, z_bottom), on the axis of revolution
#   - Traces the right exterior of the union (pillar bottom edge,
#     pillar right edge, transition into the structure, structure right
#     edge, structure top apex)
#   - Ends at (0, z_top), back on the axis
# The polygon is *implicitly* closed by the axis edge at x = 0 (the
# line from (0, z_top) straight down to (0, z_bottom)). This implicit
# closure is what an FEA preprocessor revolves about the z-axis to
# obtain the 3D solid of revolution.
#
# How the matplotlib display reuses the same output
# -------------------------------------------------
#   - ax.fill(x, z, ...)   -- closes the polygon along x = 0 and fills
#                             the interior. With edgecolor='none', the
#                             closure is not visibly stroked.
#   - ax.plot(x, z, ...)   -- draws the polyline as given (first point
#                             to last point); the closure (last->first)
#                             is naturally NOT drawn. The visible stroke
#                             is exactly the right-side outline.
#   - The dashed axis-of-revolution annotation is drawn separately in
#     the plotting cell as a display-only overlay; it is NOT part of
#     the FEA-portable outline returned by these generators.
#
# This entire cell is self-contained and can be lifted out for use in
# an FEA preprocessor (or other downstream geometry consumer) without
# importing the rest of the notebook.
# ────────────────────────────────────────────────────────────────────
N_ARC = 200


def gen_ellipse_axisym(Rxy, Rz, Rxy_p, Hp, n=N_ARC):
    """Right-half ellipsoid centered at (0, 0) merged with pillar below.
    Pillar: x in [0, Rxy_p], z in [-Hp, 0].
    Pillar's upper portion is hidden inside the ellipsoid's lower half."""
    if Rxy_p >= Rxy:
        raise ValueError(f"Pillar must be narrower than ellipse: Rxy_p={Rxy_p}, Rxy={Rxy}")
    # Crossover where pillar right edge (x = Rxy_p) meets the lower-right
    # ellipsoid edge. Ellipsoid surface satisfies x^2/Rxy^2 + z^2/Rz^2 = 1.
    # Setting x = Rxy_p and taking the negative root (lower half):
    #
    #     z_int = -Rz * sqrt(1 - (Rxy_p/Rxy)^2)
    #
    # Below z_int (toward z = -Hp), pillar is wider than ellipsoid and
    # dominates the union exterior. Above z_int, ellipsoid is wider and
    # dominates; the pillar's upper portion (z_int < z <= 0) is volumetri-
    # cally inside the ellipsoid and is hidden from the merged outline.
    z_int = -Rz * np.sqrt(1 - (Rxy_p / Rxy) ** 2)

    pillar_x = np.array([0, Rxy_p, Rxy_p])
    pillar_z = np.array([-Hp, -Hp, z_int])

    # Ellipse arc from the crossover up around to the top apex (0, Rz).
    # Parametrize x = Rxy cos(theta), z = Rz sin(theta).
    theta_start = np.arctan2(z_int / Rz, Rxy_p / Rxy)  # 4th-quadrant angle
    theta_end = np.pi / 2
    t = np.linspace(theta_start, theta_end, n)
    arc_x = Rxy * np.cos(t)
    arc_z = Rz * np.sin(t)

    return np.concatenate([pillar_x, arc_x]), np.concatenate([pillar_z, arc_z])


def gen_cone_axisym(Rxy, H, Rxy_p, Hp):
    """Right-half cone (base at z=0, apex at (0, H)) + pillar below.
    Flat-bottomed: cone and pillar meet at z = 0, no volumetric overlap."""
    x = np.array([0, Rxy_p, Rxy_p, Rxy, 0])
    z = np.array([-Hp, -Hp, 0, 0, H])
    return x, z


def gen_cap_axisym(Rxy, Rz, Rxy_p, Hp, n=N_ARC):
    """Right-half hemi-ellipse cap (base at z=0, apex at (0, Rz)) + pillar.
    Flat-bottomed: cap and pillar meet at z = 0."""
    pillar_x = np.array([0, Rxy_p, Rxy_p, Rxy])
    pillar_z = np.array([-Hp, -Hp, 0, 0])
    t = np.linspace(0, np.pi / 2, n)
    arc_x = Rxy * np.cos(t)
    arc_z = Rz * np.sin(t)
    return np.concatenate([pillar_x, arc_x]), np.concatenate([pillar_z, arc_z])


def gen_capcone_axisym(Rxy, Rz, r, Rxy_p, Hp, n=N_ARC):
    """Right-half cap-cone (umbonate, hemisphere-compound) + pillar below.
    Currently supports only Rxy == Rz."""
    if Rxy != Rz:
        raise NotImplementedError("gen_capcone_axisym currently supports only Rxy == Rz")
    R_main, R_small, d = Rxy, r, Rz
    sin_theta = (R_main - R_small) / d
    theta = np.arcsin(sin_theta)

    # Pillar + bottom transition to cap base
    pillar_x = np.array([0, Rxy_p, Rxy_p, Rxy])
    pillar_z = np.array([-Hp, -Hp, 0, 0])
    # Main hemi-arc from (Rxy, 0) at theta=0 to (R_main cos theta, R_main sin theta)
    n_main = max(int(n * theta / (np.pi / 2)), 5)
    t_main = np.linspace(0, theta, n_main)
    main_x = R_main * np.cos(t_main)
    main_z = R_main * np.sin(t_main)
    # Tangent line is implicit between main_arc end and small_arc start.
    # Small hemi-arc from (R_small cos theta, d + R_small sin theta) to (0, d + R_small).
    n_small = max(int(n * (np.pi / 2 - theta) / (np.pi / 2)), 10)
    t_small = np.linspace(theta, np.pi / 2, n_small)
    small_x = R_small * np.cos(t_small)
    small_z = d + R_small * np.sin(t_small)

    x = np.concatenate([pillar_x, main_x, small_x])
    z = np.concatenate([pillar_z, main_z, small_z])
    return x, z


def gen_doublecone_axisym(Rxy, H, Rxy_p, Hp):
    """Right-half doublecone (rhombus) centered at (0, 0) + pillar below.
    Full-rhombus vertices: (0, +/-H/2), (+/-Rxy, 0). Lower-right edge is
    the line from (0, -H/2) to (Rxy, 0); pillar partially merges into it."""
    if Rxy_p >= Rxy:
        raise ValueError(f"Pillar must be narrower than doublecone: Rxy_p={Rxy_p}, Rxy={Rxy}")
    # Crossover where pillar right edge (x = Rxy_p) meets the lower-right
    # doublecone edge. That edge runs from (0, -H/2) to (Rxy, 0) and so
    # satisfies x = Rxy * (1 + 2z/H). Setting x = Rxy_p and solving:
    #
    #     z_int = (H/2) * (Rxy_p/Rxy - 1)
    #
    # which is negative because Rxy_p < Rxy. Below z_int, pillar is
    # wider than the doublecone and dominates the union exterior.
    # Above z_int (up to z = 0), the doublecone is wider; the pillar's
    # upper portion (z_int < z <= 0) is volumetrically inside the lower
    # cone of the rhombus and is hidden from the merged outline.
    z_int = (H / 2) * (Rxy_p / Rxy - 1)
    x = np.array([0, Rxy_p, Rxy_p, Rxy, 0])
    z = np.array([-Hp, -Hp, z_int, 0, H / 2])
    return x, z


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 3D-render image loader (carries over from the full-profile notebook):
# crop the SolidWorks viewport black background to the structure's
# bounding box, then alpha-blend any surviving dark pixels toward white.
# ────────────────────────────────────────────────────────────────────

def remove_black_bg(arr, low=30, high=90):
    arr = arr.astype(float)
    luminance = arr.max(axis=2)
    alpha = np.clip((luminance - low) / max(high - low, 1), 0, 1)
    alpha = alpha[:, :, np.newaxis]
    white = np.full_like(arr, 255.0)
    out = arr * alpha + white * (1 - alpha)
    return out.clip(0, 255).astype(np.uint8)


def crop_to_content(img_path, threshold=15, margin_px=30):
    im = Image.open(img_path).convert('RGB')
    arr = np.asarray(im)
    mask = (arr.max(axis=2) > threshold)
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    if len(rows) == 0 or len(cols) == 0:
        return remove_black_bg(arr)
    r0 = max(rows.min() - margin_px, 0)
    r1 = min(rows.max() + margin_px, arr.shape[0])
    c0 = max(cols.min() - margin_px, 0)
    c1 = min(cols.max() + margin_px, arr.shape[1])
    return remove_black_bg(arr[r0:r1, c0:c1])


In [ ]:
# Geometry metadata for the 7-shape axisymmetric table.
# Each entry binds a generator call (with structure + pillar parameters)
# and carries the structure-dim and pillar-dim strings shown in the
# dimensions column.
IMG_DIR = '.'  # directory containing the 3D-render image files

SHAPES = [
    {'geoFam': 'ellipsoids', 'specID': 'N308.1',
     'label': r'ellipse, $h<r$',
     'dims_struct': r'$R_{xy}=50$, $R_z=25$',
     'dims_pillar': r'$R_{xy,p}=25$, $H_p=31.25$',
     'gen': lambda: gen_ellipse_axisym(50, 25, 25, 31.25),
     'img': 'P_oblateEllipse.PNG'},
    {'geoFam': 'ellipsoids', 'specID': 'N308.2',
     'label': r'ellipse, $h=r$',
     'dims_struct': r'$R_{xy}=R_z=50$',
     'dims_pillar': r'$R_{xy,p}=25$, $H_p=62.5$',
     'gen': lambda: gen_ellipse_axisym(50, 50, 25, 62.5),
     'img': 'P_sphere.PNG'},
    {'geoFam': 'ellipsoids', 'specID': 'N308.3',
     'label': r'ellipse, $h>r$',
     'dims_struct': r'$R_{xy}=50$, $R_z=75$',
     'dims_pillar': r'$R_{xy,p}=25$, $H_p=93.75$',
     'gen': lambda: gen_ellipse_axisym(50, 75, 25, 93.75),
     'img': 'P_prolateEllipse.PNG'},
    {'geoFam': 'cones', 'specID': 'N406',
     'label': r'triangle (cone)',
     'dims_struct': r'$R_{xy}=50$, $H=50$',
     'dims_pillar': r'$R_{xy,p}=25$, $H_p=62.5$',
     'gen': lambda: gen_cone_axisym(50, 50, 25, 62.5),
     'img': 'P_cone.PNG'},
    {'geoFam': 'caps', 'specID': 'N504',
     'label': r'hemi-ellipse',
     'dims_struct': r'$R_{xy}=R_z=50$',
     'dims_pillar': r'$R_{xy,p}=25$, $H_p=62.5$',
     'gen': lambda: gen_cap_axisym(50, 50, 25, 62.5),
     'img': 'P_cap_SLDPRT.PNG'},
    {'geoFam': 'cap_cones', 'specID': 'N1404',
     'label': r'hemisphere-compound',
     'dims_struct': r'$R_{xy}=R_z=50$, $r=25$',
     'dims_pillar': r'$R_{xy,p}=25$, $H_p=62.5$',
     'gen': lambda: gen_capcone_axisym(50, 50, 25, 25, 62.5),
     'img': 'P_cap-cone.PNG'},
    {'geoFam': 'doublecones', 'specID': '',
     'label': r'rhombus (doublecone)',
     'dims_struct': r'$R_{xy}=50$, $H=100$',
     'dims_pillar': r'$R_{xy,p}=25$, $H_p=62.5$',
     'gen': lambda: gen_doublecone_axisym(50, 100, 25, 62.5),
     'img': 'P_doublecone.PNG'},
]


In [ ]:
# Figure builder. The axis-of-revolution dashed annotation is drawn here
# (display-only) and is not part of the FEA-portable shape outline.
SCALE       = 0.013
Y_MARGIN    = 8
X_LEFT      = -3        # small left margin so the axis line is visible
X_RIGHT     = 60        # right margin past Rxy_max = 50
FILL_COLOR  = '#D9D9D9'
EDGE_COLOR  = '#1A1A1A'
EDGE_LW     = 1.5
AXIS_COLOR  = '#666666'
AXIS_LW     = 0.8
AXIS_DASHES = (4, 3)    # (dash, gap) length in points

X_RANGE = (X_LEFT, X_RIGHT)
X_SPAN  = X_RIGHT - X_LEFT

COL_FAM_W_IN    = 1.2
COL_RENDER_W_IN = X_SPAN * SCALE
COL_DIMS_W_IN   = 2.5     # wider for the 3-line dimensions block
COL_3D_W_IN     = 1.7
TOTAL_W_IN = COL_FAM_W_IN + COL_RENDER_W_IN + COL_DIMS_W_IN + COL_3D_W_IN


def _shape_extent(s):
    x, z = s['gen']()
    return z.max(), z.min()


# Reference vertical extent across the FULL SHAPES list, so multi-page
# splits use the same row scale.
_extents = [_shape_extent(s) for s in SHAPES]
MAX_HALF_EXTENT = max(max(t for t, _ in _extents), -min(b for _, b in _extents))
ROW_H_UNITS = 2 * MAX_HALF_EXTENT + 2 * Y_MARGIN


def build_figure(shapes_subset):
    """Build a table figure for the given subset of SHAPES."""
    row_h_in = ROW_H_UNITS * SCALE
    total_h_in = row_h_in * len(shapes_subset)

    fig = plt.figure(figsize=(TOTAL_W_IN, total_h_in), facecolor='white')
    gs = fig.add_gridspec(
        nrows=len(shapes_subset), ncols=4,
        width_ratios=[COL_FAM_W_IN, COL_RENDER_W_IN, COL_DIMS_W_IN, COL_3D_W_IN],
        height_ratios=[row_h_in] * len(shapes_subset),
        hspace=0.0, wspace=0.0,
        left=0.0, right=1.0, top=1.0, bottom=0.0,
    )

    if not hasattr(build_figure, '_img_cache'):
        build_figure._img_cache = {}

    def get_img(fn):
        if fn not in build_figure._img_cache:
            build_figure._img_cache[fn] = crop_to_content(f"{IMG_DIR}/{fn}")
        return build_figure._img_cache[fn]

    shown_geofam = set()
    for i, s in enumerate(shapes_subset):
        x, z = s['gen']()
        z_top, z_bottom = z.max(), z.min()
        z_offset = -(z_top + z_bottom) / 2  # center bbox at y = 0

        # Col 0: geoFam (centered; first occurrence per family group)
        ax = fig.add_subplot(gs[i, 0])
        if s['geoFam'] not in shown_geofam:
            ax.text(0.5, 0.5, s['geoFam'], ha='center', va='center',
                    fontsize=10, fontweight='bold', transform=ax.transAxes)
            shown_geofam.add(s['geoFam'])
        ax.axis('off')

        # Col 1: 2D axisymmetric merged shape
        ax = fig.add_subplot(gs[i, 1])
        z_disp = z + z_offset
        # Filled region. edgecolor='none' so the implicit x=0 closure
        # isn't stroked; the dashed axis annotation marks it instead.
        ax.fill(x, z_disp, facecolor=FILL_COLOR, edgecolor='none')
        # Right-side outline. ax.plot doesn't auto-close, so the axis
        # closure is naturally omitted from the stroke.
        ax.plot(x, z_disp, color=EDGE_COLOR, linewidth=EDGE_LW,
                solid_joinstyle='round', solid_capstyle='round')
        # Dashed axis-of-revolution annotation at x = 0 (display-only;
        # not part of the FEA-portable shape outline).
        y_lo = -MAX_HALF_EXTENT - Y_MARGIN
        y_hi =  MAX_HALF_EXTENT + Y_MARGIN
        ax.plot([0, 0], [y_lo, y_hi], color=AXIS_COLOR,
                linewidth=AXIS_LW, linestyle=(0, AXIS_DASHES))
        ax.set_xlim(*X_RANGE)
        ax.set_ylim(y_lo, y_hi)
        ax.set_aspect('equal')
        ax.axis('off')

        # Col 2: label + structure dims + pillar dims (3 lines, centered)
        ax = fig.add_subplot(gs[i, 2])
        ax.text(0.05, 0.68, s['label'], ha='left', va='center',
                fontsize=10, fontweight='bold', transform=ax.transAxes)
        ax.text(0.05, 0.50, s['dims_struct'] + r' $\mu$m',
                ha='left', va='center', fontsize=9, transform=ax.transAxes)
        ax.text(0.05, 0.32, 'pillar: ' + s['dims_pillar'] + r' $\mu$m',
                ha='left', va='center', fontsize=9, color='#444444',
                transform=ax.transAxes)
        ax.axis('off')

        # Col 3: 3D render (cleaned)
        ax = fig.add_subplot(gs[i, 3])
        ax.imshow(get_img(s['img']))
        ax.set_anchor('C')
        ax.axis('off')

    return fig


In [ ]:
# Full version: all 7 rows.
fig = build_figure(SHAPES)
fig.savefig('family_2d_geometries_axisym.png', dpi=300,
            bbox_inches='tight', facecolor='white')
print(f'Saved: family_2d_geometries_axisym.png  ({len(SHAPES)} rows)')
plt.show()


In [ ]:
# Two-page version.
# Page 1: ellipsoids (3 rows) + doublecones (1 row)
# Page 2: cones, caps, cap_cones (3 rows)

PAGE1 = [s for s in SHAPES if s['geoFam'] in ('ellipsoids', 'doublecones')]
PAGE2 = [s for s in SHAPES if s['geoFam'] not in ('ellipsoids', 'doublecones')]

fig1 = build_figure(PAGE1)
fig1.savefig('family_2d_geometries_axisym_page1.png', dpi=300,
             bbox_inches='tight', facecolor='white')
print(f'Saved: family_2d_geometries_axisym_page1.png  ({len(PAGE1)} rows)')

fig2 = build_figure(PAGE2)
fig2.savefig('family_2d_geometries_axisym_page2.png', dpi=300,
             bbox_inches='tight', facecolor='white')
print(f'Saved: family_2d_geometries_axisym_page2.png  ({len(PAGE2)} rows)')
